## PRISM PEFT Setup from Scratch
I don't trust the code of this repository anymore, and I want to replicate the results without unsloth. 

The goal of this noteboook is to creat a new PEFT setup with only HF Dependencies, Qwen 0.5B etc.

## Librarires

In [1]:
import trl
import peft
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

/home/jporras/miniconda3/envs/prism/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


In [2]:
1+1

2

## Loading model

In [3]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

In [4]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

In [5]:
tokenizer

Qwen2TokenizerFast(name_or_path='Qwen/Qwen2.5-0.5B-Instruct', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False,

## Loading dataset

In [6]:
import json
from datasets import load_dataset
with open("data/gpt_gen_formatted.json", "r") as f:
    data = print(json.loads(f.read()))

[{'conversations': [{'role': 'user', 'content': "task: I need a shovel. Is there one in the scene?Scene graph:{'objects': [{'name': 'house_1', 'coords': [-1, -1]}, {'name': 'house_2', 'coords': [-3, -1]}, {'name': 'grocery_store_1', 'coords': [-5, -1]}, {'name': 'shed_1', 'coords': [1, 3]}, {'name': 'shed_1', 'coords': [1, 5]}], 'regions': [{'name': 'example_road_1', 'coords': [-1, 0]}, {'name': 'example_road_2', 'coords': [-2, 0]}, {'name': 'field_11', 'coords': [0, 1]}, {'name': 'field_13', 'coords': [2, 3]}], 'object_connections': [['house_1', 'example_road_1'], ['house_2', 'example_road_2'], ['shed_1', 'field_11'], ['shed_2', 'field_13']], 'region_connections': [['example_road_1', 'example_road_2'], ['example_road_1', 'field_11'], ['field_11', 'field_13']], 'robot_location': 'example_road_1'}"}, {'role': 'assistant', 'content': '{\n        "primary_goal": "find a shovel for the user."        "relevant_graph": "field_11, field_13, unobserved_node(shovel)",         "reasoning": "The 

In [7]:
full_dataset = load_dataset("json", data_files=["data/gpt_gen_formatted.json"], split="train")
full_dataset

Dataset({
    features: ['conversations'],
    num_rows: 990
})

## Chat templates?

In [8]:
# Checkign out how the chat template looks like.
# Huggingface  has a default format: role and content
chat = [
  {"role": "user", "content": "Hello, how are you?"},
  {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
  {"role": "user", "content": "I'd like to show off how chat templating works!"},
]

# This qwen model uses im_start and im_end followed by the role as a template.
tokenizer.apply_chat_template(chat, tokenize=False)

"<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nHello, how are you?<|im_end|>\n<|im_start|>assistant\nI'm doing great. How can I help you today?<|im_end|>\n<|im_start|>user\nI'd like to show off how chat templating works!<|im_end|>\n"

In [9]:
full_dataset[0]['conversations']

[{'content': "task: I need a shovel. Is there one in the scene?Scene graph:{'objects': [{'name': 'house_1', 'coords': [-1, -1]}, {'name': 'house_2', 'coords': [-3, -1]}, {'name': 'grocery_store_1', 'coords': [-5, -1]}, {'name': 'shed_1', 'coords': [1, 3]}, {'name': 'shed_1', 'coords': [1, 5]}], 'regions': [{'name': 'example_road_1', 'coords': [-1, 0]}, {'name': 'example_road_2', 'coords': [-2, 0]}, {'name': 'field_11', 'coords': [0, 1]}, {'name': 'field_13', 'coords': [2, 3]}], 'object_connections': [['house_1', 'example_road_1'], ['house_2', 'example_road_2'], ['shed_1', 'field_11'], ['shed_2', 'field_13']], 'region_connections': [['example_road_1', 'example_road_2'], ['example_road_1', 'field_11'], ['field_11', 'field_13']], 'robot_location': 'example_road_1'}",
  'role': 'user'},
 {'content': '{\n        "primary_goal": "find a shovel for the user."        "relevant_graph": "field_11, field_13, unobserved_node(shovel)",         "reasoning": "The graph does not contain any shovels. H

## Manually turn ShareGPT into HF format
Not sure if this is necessary but didn't find the function outside of sloth (sloth equivalent: `standardize_sharegpt`)

Note: `trl` recommends using their `apply_chat_template` function. https://huggingface.co/docs/trl/dataset_formats



ShareGPT Format: [{role:"",content:""},...] inside a conversations key.
```python
{'content': "task: I need a shovel. Is there one in the scene?Scene graph:{'objects': [{'name': 'house_1', 'coords': [-1, -1]}, {'name': 'house_2', 'coords': [-3, -1]}, {'name': 'grocery_store_1', 'coords': [-5, -1]}, {'name': 'shed_1', 'coords': [1, 3]}, {'name': 'shed_1', 'coords': [1, 5]}], 'regions': [{'name': 'example_road_1', 'coords': [-1, 0]}, {'name': 'example_road_2', 'coords': [-2, 0]}, {'name': 'field_11', 'coords': [0, 1]}, {'name': 'field_13', 'coords': [2, 3]}], 'object_connections': [['house_1', 'example_road_1'], ['house_2', 'example_road_2'], ['shed_1', 'field_11'], ['shed_2', 'field_13']], 'region_connections': [['example_road_1', 'example_road_2'], ['example_road_1', 'field_11'], ['field_11', 'field_13']], 'robot_location': 'example_road_1'}",
   'role': 'user'},
  {'content': '{\n        "primary_goal": "find a shovel for the user."        "relevant_graph": "field_11, field_13, unobserved_node(shovel)",         "reasoning": "The graph does not contain any shovels. However, I know that the graph may be incomplete, so I will explore before providing a definitive answer. I will first map, then if needed I will add regions. There are two sheds in the scene, and shovels are often found near sheds. Therefore, for each of the sheds, I will navigate to the nearby region and map.",        "plan": "[goto(field_11), map_region(field_11), goto(field_13), map_region(field_13)]"\n}',
   'role': 'assistant'},
```


HF Formats:
```python
# Standard language modeling
{"text": "The sky is blue."}

# Conversational language modeling
{"messages": [{"role": "user", "content": "What color is the sky?"},
              {"role": "assistant", "content": "It is blue."}]}

# Standard prompt-completion
{"prompt": "The sky is",
 "completion": " blue."}

# Conversational prompt-completion
{"prompt": [{"role": "user", "content": "What color is the sky?"}],
 "completion": [{"role": "assistant", "content": "It is blue."}]}
```



In [ ]:
import trl
fd = full_dataset.map(lambda e: trl.apply_chat_template({"messages": e['conversations']},tokenizer=tokenizer,num_proc=1),num_proc=1,remove_columns=['conversations'])
fd

Dataset({
    features: ['text'],
    num_rows: 990
})

## Train

In [16]:
conversational = full_dataset.map(lambda e: {"messages": e['conversations']},num_proc=1,remove_columns=['conversations'])

In [17]:
from trl import SFTTrainer,SFTConfig

sft_config = SFTConfig(
    output_dir="outputs",
    max_steps=1,
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    assistant_only_loss=True,
)

trainer = SFTTrainer(
    model,
    args=sft_config,
    train_dataset=conversational,    
)

trainer.train()

Tokenizing train dataset:   0%|          | 0/990 [00:00<?, ? examples/s]


RuntimeError: You're using `assistant_only_loss=True`, but at least one example has no assistant tokens. This usually means the tokenizer's chat template doesn't generate assistant masks — it may be missing the `{% generation %}` keyword. Please check the template and ensure it's correctly configured to support assistant masking.

## Can we use llama3.2 3B?

In [ ]:
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3.2-3B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.2-3B-Instruct")

## Parsing Scene Graphs into Torch geometric